# 02 — Timing local em CPU do baseline ResNet-50

**Objetivo:** rodar **1 epoch completo** (não-smoke) do baseline em CPU local para **medir tempo real** e tomar decisão informada sobre Colab vs CPU vs Colab Pro.

Não usa Drive, não clona repo — roda direto da pasta local `c:\Users\thiag\TCC` com `.venv` ativo.

## Por que esse notebook existe (F-0011)

Tentativa anterior no Colab gratuito não completou 1 epoch porque o `Data/` estava sendo lido direto do Drive via symlink (cada `__getitem__` = chamada FUSE lenta). Ficou sem cota de unidades computacionais antes de qualquer resultado real. Notebook Colab vai ser corrigido (copiar Data.zip para disco local antes do treino), mas isso pode esperar a cota renovar.

Enquanto isso, este notebook responde a pergunta concreta: **quanto tempo realmente leva 1 epoch em CPU local na minha máquina?**

## Pré-requisitos

- VSCode aberto em `c:\Users\thiag\TCC`.
- Selecionar o kernel Python do `.venv` do projeto:
  - Clica no seletor de kernel no canto superior direito → **Select Another Kernel** → **Python Environments** → **`.venv (Python 3.11.9)`**.
- Confirmar com a primeira célula que está usando o `.venv` correto.

## O que esperar

Pelo extrapolado a partir do smoke test: cada epoch de train completo (1.886 batches × 32 imagens) deve levar ~3-5h em CPU. **A célula de treino mostra progresso por batch via `tqdm` no terminal embutido** — vai dar para você ver o ritmo após poucos minutos e cancelar (Stop button) se ficar inviável.

## 1. Validar ambiente

In [ ]:
import sys, os
print(f'Python : {sys.version.split()[0]}')
print(f'Executable : {sys.executable}')
print(f'CWD : {os.getcwd()}')

assert '.venv' in sys.executable, (
    'Kernel NAO esta usando o .venv do projeto. '
    'Troque o kernel no canto superior direito do VSCode.'
)
import torch, timm, torchmetrics
print(f'\ntorch       : {torch.__version__}  (cuda={torch.cuda.is_available()})')
print(f'timm        : {timm.__version__}')
print(f'torchmetrics: {torchmetrics.__version__}')

Se algum import falhar: rode no terminal (com `.venv` ativo) `pip install -r requirements.txt`. Se o `assert` do `.venv` falhar: troque o kernel no VSCode antes de continuar.

## 2. Conferir estrutura local

Espera-se manifest gerado, split versionado, e a pasta `Data/` no projeto.

In [ ]:
PROJECT_ROOT = os.getcwd()
if not PROJECT_ROOT.endswith('TCC'):
    # Notebook está em notebooks/, sobe um nível
    os.chdir(os.path.dirname(PROJECT_ROOT))
    PROJECT_ROOT = os.getcwd()
print(f'PROJECT_ROOT: {PROJECT_ROOT}')

assert os.path.isfile('results/eda/manifest.csv'), 'manifest.csv ausente'
assert os.path.isfile('experiments/splits/split_v1.json'), 'split_v1.json ausente'
assert os.path.isdir('Data/Non Demented'), 'Data/Non Demented ausente'

# Quick sanity sobre o dataset
import sys
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from src.data.dataset import OASISDataset
ds = OASISDataset(
    manifest_path='results/eda/manifest.csv',
    split_path='experiments/splits/split_v1.json',
    fold='train', label_scheme='class_3', data_root='.',
)
print(f'train: {len(ds)} slices, {ds.subject_counts()} pacientes')
print(f'classes: {ds.class_counts()}')

## 3. Calibração: tempo de 1 batch

Antes de mandar 1 epoch inteiro, medimos o custo de **1 batch** (forward+backward+optimizer step). Multiplicando pelos 1.886 batches da epoch, dá uma estimativa rápida em segundos. Se for inviável, paramos aqui e mudamos de estratégia.

In [ ]:
import time, random, numpy as np
from torch.utils.data import DataLoader
from src.data.dataset import build_dataloaders
from src.models.resnet import build_resnet50
from src.training.loop import compute_class_weights

random.seed(42); np.random.seed(42); torch.manual_seed(42)

loaders = build_dataloaders(
    manifest_path='results/eda/manifest.csv',
    split_path='experiments/splits/split_v1.json',
    label_scheme='class_3', data_root='.',
    image_size=224,
    batch_size_train=32, batch_size_eval=64,
    num_workers=0,  # Windows + CPU: workers extras pouco ajudam, costumam dar erro
)
model, info = build_resnet50(num_classes=3, pretrained=True)
device = torch.device('cpu')
model = model.to(device).train()

weights = compute_class_weights(loaders['train'], num_classes=3).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)

# Roda 3 batches: descarta o 1o (warm-up) e tira média dos 2 últimos
times = []
iter_loader = iter(loaders['train'])
for i in range(3):
    x, y = next(iter_loader)
    x, y = x.to(device), y.to(device)
    t0 = time.time()
    optimizer.zero_grad(set_to_none=True)
    logits = model(x)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()
    dt = time.time() - t0
    times.append(dt)
    print(f'  batch {i+1}: {dt:.2f}s   loss={loss.item():.3f}')

batch_time = sum(times[1:]) / len(times[1:])
n_batches = len(loaders['train'])
epoch_train_est = batch_time * n_batches
epoch_eval_est = batch_time * 0.5 * (len(loaders['val']) + len(loaders['test']))  # eval ~50% de train (sem backward)
total_per_epoch_h = (epoch_train_est + epoch_eval_est) / 3600
print(f'\n=== Estimativa ===')
print(f'  tempo medio por batch (treino)   : {batch_time:.2f}s')
print(f'  batches por epoch (treino)       : {n_batches}')
print(f'  epoch train apenas               : {epoch_train_est/60:.1f} min ({epoch_train_est/3600:.2f} h)')
print(f'  epoch train + val + test eval    : ~{total_per_epoch_h:.2f} h por epoch')
print(f'  20 epochs (sem early stopping)   : ~{20*total_per_epoch_h:.1f} h ({20*total_per_epoch_h/24:.1f} dias)')
print(f'  10 epochs (com early stopping)   : ~{10*total_per_epoch_h:.1f} h ({10*total_per_epoch_h/24:.1f} dias)')

**Pare aqui e leia a estimativa antes de prosseguir.**

- Se a estimativa para 10 epochs for **> 24h**: CPU local é inviável. Próximos passos:
  - Esperar cota Colab restaurar (1-3 dias) e usar o notebook 01 corrigido (com Data.zip → /content/).
  - Considerar Colab Pro (R$ 50-60/mês, A100/L4) → 1 epoch em ~1-2 min, treino completo em ~30 min.
  - Pedir acesso a recurso GPU da UECE (laboratório, professor).
- Se estiver entre **6-24h**: viável mas longo. Rodar 1 epoch agora pra ter ground truth (próxima célula).
- Se estiver **< 6h**: máquina é melhor que a estimativa anterior. Vale rodar 1 epoch.

## 4. Treinar 1 epoch real (opcional — só se a estimativa acima for tolerável)

Usa o entrypoint do projeto. Output salva em `experiments/runs/<id>_resnet50_local_timing/`. Progresso visível por batch via `tqdm`.

**Para abortar a qualquer momento:** clique no quadrado de Stop ao lado da célula (interrupt kernel). O melhor checkpoint salvo até ali fica preservado.

In [ ]:
import subprocess, sys
cmd = [
    sys.executable, '-m', 'src.training.run',
    '--epochs', '1',
    '--batch-size', '32',
    '--batch-size-eval', '64',
    '--num-workers', '0',
    '--run-name', 'resnet50_local_timing',
]
print('Comando:', ' '.join(cmd))
print('Inicio:', __import__('datetime').datetime.now().strftime('%H:%M:%S'))
subprocess.run(cmd, check=False)
print('Fim:', __import__('datetime').datetime.now().strftime('%H:%M:%S'))

## 5. Análise do tempo medido

Após o treino terminar (ou ser abortado), lê o `history.json` da run e compara o tempo medido com a estimativa da célula 3.

In [ ]:
import json
from pathlib import Path

runs_dir = Path('experiments/runs')
matching = sorted([d for d in runs_dir.iterdir() if d.is_dir() and 'local_timing' in d.name])
assert matching, 'Nenhuma run local_timing encontrada (executou a celula 4?)'
run = matching[-1]
print(f'Run: {run.name}')

hist_path = run / 'history.json'
if hist_path.exists():
    history = json.loads(hist_path.read_text(encoding='utf-8'))
    if history:
        h = history[0]
        print(f'\n=== Epoch 1 (medido) ===')
        print(f'  epoch_time : {h["epoch_time_seconds"]:.0f}s  ({h["epoch_time_seconds"]/60:.1f} min, {h["epoch_time_seconds"]/3600:.2f} h)')
        print(f'  train_loss : {h["train_loss"]:.4f}')
        print(f'  val_loss   : {h["val_loss"]:.4f}')
        print(f'  val_bal_acc: {h["val_metrics"]["balanced_accuracy"]:.4f}')
        print(f'  val_macro_f1: {h["val_metrics"]["macro_f1"]:.4f}')
        print(f'  val_auc    : {h["val_metrics"]["auc_macro"]}')
        print(f'\n=== Extrapolacao ===')
        et = h['epoch_time_seconds']
        print(f'  10 epochs : {10*et/3600:.1f}h')
        print(f'  20 epochs : {20*et/3600:.1f}h')
    else:
        print('history.json vazio (epoch 1 nao terminou)')
else:
    print('history.json nao existe ainda; epoch foi abortada antes do final.')

## 6. Decisão

Com o tempo real medido, escolha um caminho:

1. **CPU local é viável** (< 10h para treino completo) → rodar `python -m src.training.run --epochs 20` no terminal e seguir.
2. **CPU local é marginal** (10-24h) → priorize Colab assim que cota renovar (1-3 dias); use o notebook 01 (com correção de Data.zip que vou aplicar antes de você usá-lo de novo).
3. **CPU local é inviável** (> 24h) → Colab Pro vira investimento que se paga. 1 mês ≈ R$50-60 → A100 fecha treino em < 30 min.

Tudo medido fica em `experiments/runs/<id>_resnet50_local_timing/` — comparável com runs futuras de Colab para validar consistência.